# RETFound-Green Fine-Tuning Pipeline
## Hypertension Risk Detection from Retinal Fundus Images

**Project context:** This notebook trains the *first-generation foundation model*. The model is a fine-tuned **RETFound-Green** backbone that outputs a **calibrated hypertension-likelihood score (0–100%)** from a single fundus photograph. This model will later become the **initial global model** distributed to client devices in a federated learning loop, so everything here (input size, preprocessing, export formats, checkpoint format) is chosen to be compatible with on-device inference on a smartphone/tablet.

**Pipeline stages covered:**

1. Environment setup & reproducibility
2. Data ingestion & directory/metadata contract
3. Exploratory data analysis (EDA)
4. Automated image quality assessment (OpenCV) - gate before training (done by frontend - Obi)
5. Retinal-specific preprocessing (FOV crop, CLAHE, normalization)
6. Patient-level, site-aware, leakage-safe dataset splitting
7. Augmentation pipeline (retina-safe)
8. PyTorch `Dataset` / `DataLoader`
9. Model adaptation — loading pretrained RETFound-Green weights + new head
10. Loss function & class-imbalance handling
11. Training loop (AMP, scheduler, early stopping, checkpointing)
12. Evaluation (discrimination, calibration, subgroup fairness, Grad-CAM)
13. Probability calibration - final 0–100% likelihood score
14. Model export (TorchScript, ONNX, TFLite, Core ML) + quantization
15. Model card / artifact packaging for the federated server


## 1. Environment Setup

In [ ]:
# %%capture
# Uncomment to install in a fresh environment (Colab / new venv).
# Pinned loosely — adjust to match your CUDA version for torch.
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q timm==1.0.9 opencv-python-headless albumentations==1.4.15 \
    scikit-learn pandas numpy matplotlib seaborn tqdm \
    onnx onnxruntime onnxscript \
    grad-cam==1.5.4 \
    huggingface_hub \
    coremltools \
    tensorflow onnx-tf \
    pyyaml


In [ ]:
import os
import io
import json
import math
import random
import shutil
import hashlib
import dataclasses
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional, List, Tuple, Dict, Any

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, brier_score_loss, f1_score
)
from sklearn.isotonic import IsotonicRegression

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", DEVICE)

### Central Config

Every path, hyperparameter and toggle lives here so the run is reproducible and diffable across
experiments. Save this dict alongside every checkpoint (done automatically in Section 11).


In [ ]:
@dataclass
class Config:
    # paths
    project_root: str = "./retina_htn_project"
    raw_images_dir: str = "./data/raw_images"          # collected/proprietary fundus photos
    metadata_csv: str = "./data/metadata.csv"          # see contract in Section 2
    processed_dir: str = "./data/processed"            # cache of preprocessed images
    checkpoints_dir: str = "./checkpoints"
    exports_dir: str = "./exports"
    reports_dir: str = "./reports"

    # data contract
    label_col: str = "hypertension_label"     # 0/1, clinician/measured-BP derived
    patient_id_col: str = "patient_id"
    site_id_col: str = "site_id"              # important: federated clients == sites
    image_path_col: str = "image_path"

    # image / model
    image_size: int = 224                     # RETFound-Green default input res
    in_channels: int = 3
    retfound_hf_repo: str = "YukunZhou/RETFound_mae_green"   # placeholder: confirm exact
    retfound_variant: str = "vit_large_patch16"              # RETFound-Green backbone family
    freeze_backbone_epochs: int = 5           # linear-probe warmup before unfreezing
    dropout: float = 0.2

    # splits
    test_size: float = 0.15
    val_size: float = 0.15                    # of the remaining train+val pool
    n_cv_folds: int = 5                       # for StratifiedGroupKFold sanity check

    # quality gate
    min_sharpness_laplacian_var: float = 40.0
    min_illumination_mean: float = 30.0
    max_illumination_mean: float = 225.0
    min_fov_ratio: float = 0.55               # fraction of frame that is retina, not black border

    # training
    batch_size: int = 32
    num_workers: int = 4
    epochs: int = 40
    lr_head: float = 3e-4
    lr_backbone: float = 3e-5
    weight_decay: float = 1e-4
    label_smoothing: float = 0.05
    early_stopping_patience: int = 8
    mixed_precision: bool = True
    grad_accum_steps: int = 1

    # reproducibility
    seed: int = 42

CFG = Config()
for d in [CFG.project_root, CFG.processed_dir, CFG.checkpoints_dir, CFG.exports_dir, CFG.reports_dir]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(json.dumps(asdict(CFG), indent=2))

## 2. Data Ingestion & Metadata Contract (I'd still need to update this section when we collect our data)

**Expected directory layout:**

```
data/
├── raw_images/
│   ├── site01_p0001_OD.jpg
│   ├── site01_p0001_OS.jpg
│   └── ...
└── metadata.csv
```

**`metadata.csv` contract** (one row per image, not per patient, a patient may contribute both eyes):

| column | type | notes |
|---|---|---|
| `patient_id` | str | stable pseudonymous ID — used for grouped splitting |
| `image_path` | str | relative path under `raw_images_dir` |
| `eye` | str | `OD` / `OS` |
| `site_id` | str | collection site — doubles as the future **federated client ID** |
| `hypertension_label` | int (0/1) | ground truth: measured BP ≥140/90 or clinician diagnosis |
| `systolic_bp`, `diastolic_bp` | float, optional | if available, enables a regression head later |
| `age`, `sex` | optional | needed for subgroup fairness evaluation in Section 12.3 |
| `capture_device` | str, optional | fundus camera model — useful for domain-shift analysis |
| `consent_id` | str | reference to data-governance/consent record — **never store PII here** |

Load and sanity-check it below.


In [ ]:
meta = pd.read_csv(CFG.metadata_csv)

required_cols = [CFG.patient_id_col, CFG.image_path_col, CFG.site_id_col, CFG.label_col]
missing = [c for c in required_cols if c not in meta.columns]
assert not missing, f"metadata.csv is missing required columns: {missing}"

# Resolve to absolute paths and verify every file actually exists on disk.
meta["abs_path"] = meta[CFG.image_path_col].apply(lambda p: str(Path(CFG.raw_images_dir) / p))
meta["exists"] = meta["abs_path"].apply(os.path.exists)
n_missing_files = (~meta["exists"]).sum()
if n_missing_files:
    print(f"{n_missing_files} rows reference image files that do not exist, dropping them.")
    meta = meta[meta["exists"]].reset_index(drop=True)

# Drop exact duplicate images (same patient+eye+site captured twice by mistake).
dup_key = [CFG.patient_id_col, "eye", CFG.site_id_col]
dup_key = [c for c in dup_key if c in meta.columns]
before = len(meta)
meta = meta.drop_duplicates(subset=dup_key or None).reset_index(drop=True)
print(f"Dropped {before - len(meta)} duplicate rows.")

print(f"Total usable images: {len(meta)} | Unique patients: {meta[CFG.patient_id_col].nunique()} "
      f"| Sites: {meta[CFG.site_id_col].nunique()}")
meta.head()

## 3. Exploratory Data Analysis

Check class balance, per-site distribution, and patient-level image counts *before* investing compute in training — an undetected imbalance or a single dominant site here will quietly bias the federated global model later.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

meta[CFG.label_col].value_counts().sort_index().plot(kind="bar", ax=axes[0], color=["#4C72B0", "#C44E52"])
axes[0].set_title("Label balance (0 = normal, 1 = hypertensive)")
axes[0].set_xlabel(""); axes[0].set_ylabel("count")

meta[CFG.site_id_col].value_counts().plot(kind="bar", ax=axes[1], color="#55A868")
axes[1].set_title("Images per site (future federated clients)")
axes[1].tick_params(axis="x", rotation=45)

images_per_patient = meta.groupby(CFG.patient_id_col).size()
sns.histplot(images_per_patient, bins=range(1, images_per_patient.max() + 2), ax=axes[2], color="#8172B2")
axes[2].set_title("Images per patient")

plt.tight_layout()
plt.savefig(f"{CFG.reports_dir}/eda_overview.png", dpi=150)
plt.show()

# Cross-tab: does class balance vary a lot by site? (flags domain-shift / label-leak-by-site risk)
print(pd.crosstab(meta[CFG.site_id_col], meta[CFG.label_col], normalize="index").round(2))

if "age" in meta.columns:
    print("\nAge stats by label:")
    print(meta.groupby(CFG.label_col)["age"].describe()[["mean", "std", "min", "max"]])

## 4. Automated Image Quality Assessment (temporary, might remove this since Obi already has one from the frontend)

Checks implemented:
- **Sharpness**: variance of the Laplacian (low variance ⇒ blur)
- **Illumination**: mean pixel intensity in a sane range (too dark/bright ⇒ exposure issue)
- **Field-of-view (FOV) ratio**: fraction of the frame that isn't black border (poor centering / zoom)
- **Vessel-visibility proxy**: edge density inside the FOV mask, a cheap stand-in for "can we see vasculature at all"

In [ ]:
def compute_quality_metrics(img_bgr: np.ndarray) -> Dict[str, float]:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # 1. Sharpness
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()

    # 2. Illumination
    illumination = float(gray.mean())

    # 3. FOV mask + ratio (retina is the bright, roughly circular region vs. black background)
    _, fov_mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    fov_mask = cv2.morphologyEx(fov_mask, cv2.MORPH_OPEN, np.ones((9, 9), np.uint8))
    fov_ratio = float(fov_mask.sum() / 255) / (gray.shape[0] * gray.shape[1])

    # 4. Vessel-visibility proxy: edge density within the FOV
    edges = cv2.Canny(gray, 40, 120)
    edges_in_fov = cv2.bitwise_and(edges, edges, mask=fov_mask)
    vessel_proxy = float(edges_in_fov.sum() / 255) / max(fov_mask.sum() / 255, 1)

    return {
        "sharpness": sharpness,
        "illumination": illumination,
        "fov_ratio": fov_ratio,
        "vessel_proxy": vessel_proxy,
    }


def passes_quality_gate(metrics: Dict[str, float], cfg: Config = CFG) -> Tuple[bool, List[str]]:
    reasons = []
    if metrics["sharpness"] < cfg.min_sharpness_laplacian_var:
        reasons.append("blurry")
    if not (cfg.min_illumination_mean <= metrics["illumination"] <= cfg.max_illumination_mean):
        reasons.append("bad_illumination")
    if metrics["fov_ratio"] < cfg.min_fov_ratio:
        reasons.append("poor_fov_centering")
    return (len(reasons) == 0), reasons

In [ ]:
# Run the gate across the dataset once and cache results

quality_records = []
for _, row in tqdm(meta.iterrows(), total=len(meta), desc="Quality-scoring images"):
    img = cv2.imread(row["abs_path"])
    if img is None:
        quality_records.append({"ok": False, "reasons": "unreadable"})
        continue
    m = compute_quality_metrics(img)
    ok, reasons = passes_quality_gate(m)
    quality_records.append({**m, "ok": ok, "reasons": ",".join(reasons)})

quality_df = pd.DataFrame(quality_records)
meta = pd.concat([meta.reset_index(drop=True), quality_df], axis=1)

print(meta["ok"].value_counts())
print("\nRejection reasons breakdown:")
print(meta.loc[~meta["ok"], "reasons"].value_counts())

meta_clean = meta[meta["ok"]].reset_index(drop=True)
meta_clean.to_csv(f"{CFG.reports_dir}/metadata_quality_filtered.csv", index=False)
print(f"\nKept {len(meta_clean)} / {len(meta)} images after quality gate.")


In [ ]:
# Visual spot-check: sample a few passed / failed images side by side.

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, (ok_flag, title) in enumerate([(True, "PASS"), (False, "FAIL")]):
    sample = meta[meta["ok"] == ok_flag].sample(min(4, (meta["ok"] == ok_flag).sum()), random_state=SEED)
    for j, (_, row) in enumerate(sample.iterrows()):
        img = cv2.cvtColor(cv2.imread(row["abs_path"]), cv2.COLOR_BGR2RGB)
        axes[i, j].imshow(img)
        axes[i, j].set_title(f"{title}: {row.get('reasons', '')}", fontsize=9)
        axes[i, j].axis("off")
plt.tight_layout()
plt.savefig(f"{CFG.reports_dir}/quality_gate_samples.png", dpi=150)
plt.show()


## 5. Retinal-Specific Preprocessing

Standard pipeline used across most fundus foundation models (including RETFound):

1. **Circular FOV crop** — crop tightly to the circular retina region, removing black border
2. **CLAHE** on the green/luminance channel, fundus vasculature has the most contrast in green
3. **Resize** to the model's native input resolution (224×224 for RETFound-Green ViT)
4. **Normalize** using RETFound's pretraining mean/std

This is implemented as a standalone function so it's identical in training, evaluation, and later in the on-device inference code.

In [ ]:
# RETFound normalization stats

RETFOUND_MEAN = (0.485, 0.456, 0.406)
RETFOUND_STD = (0.229, 0.224, 0.225)


def crop_to_fov(img_bgr: np.ndarray, margin_px: int = 2) -> np.ndarray:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((9, 9), np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return img_bgr
    x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
    x0, y0 = max(x - margin_px, 0), max(y - margin_px, 0)
    x1, y1 = min(x + w + margin_px, img_bgr.shape[1]), min(y + h + margin_px, img_bgr.shape[0])
    
    return img_bgr[y0:y1, x0:x1]


def apply_clahe(img_bgr: np.ndarray, clip_limit: float = 2.5, tile_grid_size=(8, 8)) -> np.ndarray:
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_eq = clahe.apply(l)
    
    return cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2BGR)


def preprocess_fundus(img_bgr: np.ndarray, size: int = CFG.image_size) -> np.ndarray:
    '''Full deterministic preprocessing: crop → CLAHE → resize → RGB. Returns uint8 RGB array.'''
    img = crop_to_fov(img_bgr)
    img = apply_clahe(img)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

In [ ]:
# Visualize the before/after effect of preprocessing on a few samples.

fig, axes = plt.subplots(3, 2, figsize=(8, 12))
sample = meta_clean.sample(3, random_state=SEED)
for i, (_, row) in enumerate(sample.iterrows()):
    raw = cv2.cvtColor(cv2.imread(row["abs_path"]), cv2.COLOR_BGR2RGB)
    processed = preprocess_fundus(cv2.imread(row["abs_path"]))
    axes[i, 0].imshow(raw); axes[i, 0].set_title("raw"); axes[i, 0].axis("off")
    axes[i, 1].imshow(processed); axes[i, 1].set_title("preprocessed"); axes[i, 1].axis("off")
plt.tight_layout()
plt.savefig(f"{CFG.reports_dir}/preprocessing_before_after.png", dpi=150)
plt.show()

In [ ]:
# Cache preprocessed images to disk once — avoids re-running CLAHE/crop every epoch.

def cache_processed_images(df: pd.DataFrame, out_dir: str) -> pd.DataFrame:
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    processed_paths = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Caching preprocessed images"):
        out_path = Path(out_dir) / (Path(row["abs_path"]).stem + ".png")
        if not out_path.exists():
            img = cv2.imread(row["abs_path"])
            processed = preprocess_fundus(img)
            cv2.imwrite(str(out_path), cv2.cvtColor(processed, cv2.COLOR_RGB2BGR))
        processed_paths.append(str(out_path))
    df = df.copy()
    df["processed_path"] = processed_paths
    return df

meta_clean = cache_processed_images(meta_clean, CFG.processed_dir)
meta_clean.to_csv(f"{CFG.reports_dir}/metadata_final.csv", index=False)

## 6. Patient-Level, Leakage-Safe Dataset Splitting

Two methods are used here

- *Option A (i.i.d. split):* mix all sites into train/val/test, measures in-distribution accuracy.
- *Option B (leave-one-site-out):* hold out an entire site for test — measures how well the model generalizes to a **new federated client's population**.

Option B is used as the default.

In [2]:
def stratified_group_split(df: pd.DataFrame, cfg: Config = CFG, seed: int = SEED):
    '''Option A: i.i.d. patient-grouped stratified split.'''
    groups = df[cfg.patient_id_col]
    y = df[cfg.label_col]

    sgkf = StratifiedGroupKFold(n_splits=int(round(1 / cfg.test_size)), shuffle=True, random_state=seed)
    trainval_idx, test_idx = next(sgkf.split(df, y, groups))
    trainval_df, test_df = df.iloc[trainval_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)

    sgkf2 = StratifiedGroupKFold(n_splits=int(round(1 / cfg.val_size)), shuffle=True, random_state=seed)
    tr_idx, val_idx = next(sgkf2.split(trainval_df, trainval_df[cfg.label_col], trainval_df[cfg.patient_id_col]))
    train_df, val_df = trainval_df.iloc[tr_idx].reset_index(drop=True), trainval_df.iloc[val_idx].reset_index(drop=True)
    return train_df, val_df, test_df


def leave_one_site_out_split(df: pd.DataFrame, cfg: Config = CFG, seed: int = SEED, held_out_site: Optional[str] = None):
    '''Option B (recommended): hold out one entire site as the test set; grouped+stratified
    split of the remaining sites for train/val.'''
    site_counts = df[cfg.site_id_col].value_counts()
    held_out_site = held_out_site or site_counts.index[len(site_counts) // 2]  # pick a mid-sized site by default
    test_df = df[df[cfg.site_id_col] == held_out_site].reset_index(drop=True)
    remaining = df[df[cfg.site_id_col] != held_out_site].reset_index(drop=True)

    sgkf = StratifiedGroupKFold(n_splits=int(round(1 / cfg.val_size)), shuffle=True, random_state=seed)
    tr_idx, val_idx = next(sgkf.split(remaining, remaining[cfg.label_col], remaining[cfg.patient_id_col]))
    train_df, val_df = remaining.iloc[tr_idx].reset_index(drop=True), remaining.iloc[val_idx].reset_index(drop=True)
    print(f"Held-out test site: {held_out_site} ({len(test_df)} images)")
    return train_df, val_df, test_df


# Choose the split strategy here:
USE_LEAVE_ONE_SITE_OUT = True
if USE_LEAVE_ONE_SITE_OUT and meta_clean[CFG.site_id_col].nunique() > 1:
    train_df, val_df, test_df = leave_one_site_out_split(meta_clean)
else:
    train_df, val_df, test_df = stratified_group_split(meta_clean)

# Hard assertion: zero patient overlap across splits.
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    overlap = set(eval(f"{a}_df")[CFG.patient_id_col]) & set(eval(f"{b}_df")[CFG.patient_id_col])
    assert not overlap, f"Patient leakage detected between {a} and {b}: {overlap}"

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("Label balance —",
      "train:", train_df[CFG.label_col].mean().round(3),
      "val:", val_df[CFG.label_col].mean().round(3),
      "test:", test_df[CFG.label_col].mean().round(3))

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split_df.to_csv(f"{CFG.reports_dir}/split_{split_name}.csv", index=False)

NameError: name 'CFG' is not defined

## 7. Augmentation (Retina-Safe)

In [ ]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=20, border_mode=cv2.BORDER_CONSTANT, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussNoise(var_limit=(5.0, 20.0), p=0.2),
    A.OpticalDistortion(distort_limit=0.05, shift_limit=0.05, p=0.2),
    A.CoarseDropout(max_holes=4, max_height=16, max_width=16, p=0.2),  # simulates lens artifacts/eyelash occlusion
    A.Normalize(mean=RETFOUND_MEAN, std=RETFOUND_STD),
    ToTensorV2(),
])

eval_transform = A.Compose([
    A.Normalize(mean=RETFOUND_MEAN, std=RETFOUND_STD),
    ToTensorV2(),
])

## 8. PyTorch Dataset & DataLoaders

Images were already cached as preprocessed PNGs in Section 5, so `__getitem__` only needs to load and augment, this keeps epoch time down.

In [ ]:
class FundusHypertensionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, cfg: Config, transform=None):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.cvtColor(cv2.imread(row["processed_path"]), cv2.COLOR_BGR2RGB)
        label = float(row[self.cfg.label_col])

        if self.transform:
            img = self.transform(image=img)["image"]

        return {
            "image": img,
            "label": torch.tensor(label, dtype=torch.float32),
            "patient_id": row[self.cfg.patient_id_col],
            "site_id": row[self.cfg.site_id_col],
        }


train_ds = FundusHypertensionDataset(train_df, CFG, transform=train_transform)
val_ds = FundusHypertensionDataset(val_df, CFG, transform=eval_transform)
test_ds = FundusHypertensionDataset(test_df, CFG, transform=eval_transform)

# Class imbalance handling via weighted sampling (in addition to loss-level weighting in Sec. 10).
class_counts = train_df[CFG.label_col].value_counts().to_dict()
sample_weights = train_df[CFG.label_col].map(lambda y: 1.0 / class_counts[y]).values
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, sampler=sampler,
                           num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False,
                         num_workers=CFG.num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False,
                          num_workers=CFG.num_workers, pin_memory=True)

print(f"Batches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}")

## 9. Model Adaptation: Loading Pretrained RETFound-Green

In [ ]:
from huggingface_hub import hf_hub_download

def download_retfound_weights(cfg: Config = CFG) -> str:
    '''Downloads the RETFound-Green pretrained checkpoint. 
    Adjust filename to match the official release.'''
    local_path = hf_hub_download(
        repo_id=cfg.retfound_hf_repo,
        filename="RETFound_mae_green.pth",   # confirm exact filename on the model's HF page
        cache_dir=str(Path(cfg.project_root) / "hf_cache"),
    )
    print("Downloaded RETFound-Green weights to:", local_path)
    return local_path

# retfound_ckpt_path = download_retfound_weights(CFG)
retfound_ckpt_path = "./weights/RETFound_mae_green.pth"  # or point directly at a manually downloaded file


In [ ]:
class RETFoundHypertensionModel(nn.Module):
    '''RETFound-Green ViT encoder + a small MLP head producing a single hypertension-risk logit.'''

    def __init__(self, cfg: Config, checkpoint_path: str):
        super().__init__()
        self.cfg = cfg

        # Build a bare ViT matching RETFound's architecture (no classifier head from timm).
        self.backbone = timm.create_model(
            cfg.retfound_variant,
            img_size=cfg.image_size,
            num_classes=0,       # strip timm's head — we attach our own below
            pretrained=False,
        )
        embed_dim = self.backbone.num_features

        state_dict = self._load_retfound_state_dict(checkpoint_path)
        missing, unexpected = self.backbone.load_state_dict(state_dict, strict=False)
        print(f"Loaded RETFound weights. Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}")
        if missing:
            print("  (missing keys are expected for the MAE decoder / mask tokens, which we discard)")

        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(cfg.dropout),
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(cfg.dropout / 2),
            nn.Linear(128, 1),   # single logit; sigmoid applied in loss/inference
        )

    @staticmethod
    def _load_retfound_state_dict(checkpoint_path: str) -> Dict[str, torch.Tensor]:
        ckpt = torch.load(checkpoint_path, map_location="cpu")
        state_dict = ckpt.get("model", ckpt.get("state_dict", ckpt))
        # Strip common prefixes (e.g. "encoder.", "module.") that vary by release.
        cleaned = {}
        for k, v in state_dict.items():
            k2 = k.replace("module.", "").replace("encoder.", "")
            if k2.startswith("decoder") or "mask_token" in k2:
                continue  # MAE decoder isn't needed for downstream classification
            cleaned[k2] = v
        return cleaned

    def set_backbone_trainable(self, trainable: bool):
        for p in self.backbone.parameters():
            p.requires_grad = trainable

    def forward(self, x):
        feats = self.backbone(x)          # (B, embed_dim) — timm pools CLS/patch tokens internally
        logit = self.head(feats).squeeze(-1)
        return logit


model = RETFoundHypertensionModel(CFG, retfound_ckpt_path).to(DEVICE)
model.set_backbone_trainable(False)  # start frozen for the linear-probe warmup phase

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {n_params:,} | Trainable (frozen-backbone phase): {n_trainable:,}")

## 10. Loss Function & Class-Imbalance Handling

Binary cross-entropy with logits, plus **class weighting** (belt-and-braces alongside the
`WeightedRandomSampler` from Section 8, sampling addresses batch composition, loss weighting
addresses gradient magnitude) and light label smoothing to reduce overconfidence.


In [ ]:
pos_weight_value = (train_df[CFG.label_col] == 0).sum() / max((train_df[CFG.label_col] == 1).sum(), 1)
pos_weight = torch.tensor([pos_weight_value], device=DEVICE)
print("pos_weight (for BCE):", pos_weight_value)

def smooth_labels(y, smoothing=CFG.label_smoothing):
    return y * (1 - smoothing) + 0.5 * smoothing

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

## 11. Training Loop

Two-phase schedule, mixed precision, cosine LR decay, early stopping on validation AUROC, and a
checkpoint saved every time validation AUROC improves. Metrics are logged to a CSV each epoch.

In [ ]:
def make_optimizer(model, cfg: Config, backbone_unfrozen: bool):
    param_groups = [{"params": model.head.parameters(), "lr": cfg.lr_head}]
    if backbone_unfrozen:
        param_groups.append({"params": model.backbone.parameters(), "lr": cfg.lr_backbone})
    return torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay)


@torch.no_grad()
def evaluate(model, loader, criterion) -> Dict[str, float]:
    model.eval()
    all_logits, all_labels = [], []
    total_loss = 0.0
    for batch in loader:
        imgs = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)
        logits = model(imgs)
        loss = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs = 1 / (1 + np.exp(-logits))

    auroc = roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else float("nan")
    auprc = average_precision_score(labels, probs) if len(np.unique(labels)) > 1 else float("nan")
    preds = (probs >= 0.5).astype(int)
    f1 = f1_score(labels, preds, zero_division=0)

    return {"loss": total_loss / len(loader.dataset), "auroc": auroc, "auprc": auprc, "f1": f1,
            "probs": probs, "labels": labels}


def train():
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.mixed_precision)
    history = []
    best_auroc = -np.inf
    epochs_no_improve = 0
    backbone_unfrozen = False
    optimizer = make_optimizer(model, CFG, backbone_unfrozen)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)

    for epoch in range(1, CFG.epochs + 1):
        # Phase transition: unfreeze backbone after the linear-probe warmup.
        if epoch == CFG.freeze_backbone_epochs + 1 and not backbone_unfrozen:
            print(f"Epoch {epoch}: unfreezing backbone, switching to two-LR optimizer.")
            model.set_backbone_trainable(True)
            backbone_unfrozen = True
            optimizer = make_optimizer(model, CFG, backbone_unfrozen)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs - epoch + 1)

        model.train()
        running_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{CFG.epochs}", leave=False)):
            imgs = batch["image"].to(DEVICE, non_blocking=True)
            labels = smooth_labels(batch["label"].to(DEVICE, non_blocking=True))

            with torch.cuda.amp.autocast(enabled=CFG.mixed_precision):
                logits = model(imgs)
                loss = criterion(logits, labels) / CFG.grad_accum_steps

            scaler.scale(loss).backward()
            if (step + 1) % CFG.grad_accum_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            running_loss += loss.item() * imgs.size(0) * CFG.grad_accum_steps

        scheduler.step()
        train_loss = running_loss / len(train_loader.dataset)
        val_metrics = evaluate(model, val_loader, criterion)

        row = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_metrics["loss"],
               "val_auroc": val_metrics["auroc"], "val_auprc": val_metrics["auprc"],
               "val_f1": val_metrics["f1"], "lr_head": optimizer.param_groups[0]["lr"]}
        history.append(row)
        print(f"[{epoch:02d}] train_loss={train_loss:.4f} val_loss={val_metrics['loss']:.4f} "
              f"val_auroc={val_metrics['auroc']:.4f} val_f1={val_metrics['f1']:.4f}")

        if val_metrics["auroc"] > best_auroc:
            best_auroc = val_metrics["auroc"]
            epochs_no_improve = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "config": asdict(CFG),
                "epoch": epoch,
                "val_auroc": best_auroc,
            }, f"{CFG.checkpoints_dir}/best_model.pt")
            print(f"  ↳ new best (AUROC={best_auroc:.4f}), checkpoint saved.")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= CFG.early_stopping_patience:
                print(f"Early stopping at epoch {epoch} (no improvement for {CFG.early_stopping_patience} epochs).")
                break

    pd.DataFrame(history).to_csv(f"{CFG.reports_dir}/training_history.csv", index=False)
    return history

history = train()

In [ ]:
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="train_loss")
axes[0].plot(hist_df["epoch"], hist_df["val_loss"], label="val_loss")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(hist_df["epoch"], hist_df["val_auroc"], label="val_auroc", color="green")
axes[1].plot(hist_df["epoch"], hist_df["val_f1"], label="val_f1", color="orange")
axes[1].set_title("Validation metrics"); axes[1].legend()
plt.tight_layout()
plt.savefig(f"{CFG.reports_dir}/training_curves.png", dpi=150)
plt.show()

## 12. Evaluation on the Held-Out Test Set

Load the best checkpoint and evaluate thoroughly.

In [ ]:
best_ckpt = torch.load(f"{CFG.checkpoints_dir}/best_model.pt", map_location=DEVICE)
model.load_state_dict(best_ckpt["model_state_dict"])
model.eval()
print(f"Loaded best checkpoint from epoch {best_ckpt['epoch']} (val_auroc={best_ckpt['val_auroc']:.4f})")

test_metrics = evaluate(model, test_loader, criterion)
print(f"TEST — AUROC: {test_metrics['auroc']:.4f} | AUPRC: {test_metrics['auprc']:.4f} | F1: {test_metrics['f1']:.4f}")

### ROC / PR curves, confusion matrix, calibration curve

In [ ]:
probs, labels = test_metrics["probs"], test_metrics["labels"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

fpr, tpr, _ = roc_curve(labels, probs)
axes[0].plot(fpr, tpr, label=f"AUROC={test_metrics['auroc']:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_title("ROC curve"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].legend()

precision, recall, _ = precision_recall_curve(labels, probs)
axes[1].plot(recall, precision, label=f"AUPRC={test_metrics['auprc']:.3f}")
axes[1].set_title("Precision-Recall curve"); axes[1].legend()

cm = confusion_matrix(labels, (probs >= 0.5).astype(int))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[2],
            xticklabels=["Pred: Normal", "Pred: HTN"], yticklabels=["True: Normal", "True: HTN"])
axes[2].set_title("Confusion matrix @ 0.5 threshold")

plt.tight_layout()
plt.savefig(f"{CFG.reports_dir}/test_curves.png", dpi=150)
plt.show()

# Sensitivity / specificity at the default threshold
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
print(f"At threshold=0.5 → Sensitivity: {sensitivity:.3f} | Specificity: {specificity:.3f}")


### Calibration: Reliability Diagram & Brier Score

This is the section that directly determines whether the "% likelihood" shown is *trustworthy*.

In [ ]:
def reliability_diagram(probs, labels, n_bins=10, ax=None):
    bins = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.digitize(probs, bins) - 1
    bin_accs, bin_confs, bin_counts = [], [], []
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            continue
        bin_accs.append(labels[mask].mean())
        bin_confs.append(probs[mask].mean())
        bin_counts.append(mask.sum())

    ax = ax or plt.gca()
    ax.plot([0, 1], [0, 1], "--", color="gray", label="perfect calibration")
    ax.plot(bin_confs, bin_accs, marker="o", label="model")
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed frequency")
    ax.set_title("Reliability diagram"); ax.legend()

    ece = np.sum([c / len(probs) * abs(a - conf) for a, conf, c in zip(bin_accs, bin_confs, bin_counts)])
    return ece

fig, ax = plt.subplots(figsize=(5, 5))
ece_before = reliability_diagram(probs, labels, ax=ax)
plt.tight_layout()
plt.savefig(f"{CFG.reports_dir}/calibration_before.png", dpi=150)
plt.show()

brier = brier_score_loss(labels, probs)
print(f"Expected Calibration Error (ECE): {ece_before:.4f} | Brier score: {brier:.4f}")

### Subgroup Fairness

In [ ]:
def subgroup_report(test_df, probs, labels, group_col):
    df = test_df.copy()
    df["prob"] = probs
    df["label"] = labels
    rows = []
    for g, sub in df.groupby(group_col):
        if sub["label"].nunique() < 2:
            auroc = float("nan")
        else:
            auroc = roc_auc_score(sub["label"], sub["prob"])
        preds = (sub["prob"] >= 0.5).astype(int)
        fn_rate = ((preds == 0) & (sub["label"] == 1)).sum() / max((sub["label"] == 1).sum(), 1)
        rows.append({group_col: g, "n": len(sub), "auroc": auroc, "false_negative_rate": fn_rate})
    return pd.DataFrame(rows).sort_values("n", ascending=False)

print("By site:")
display(subgroup_report(test_df, probs, labels, CFG.site_id_col))

if "sex" in test_df.columns:
    print("\nBy sex:")
    display(subgroup_report(test_df, probs, labels, "sex"))

if "age" in test_df.columns:
    test_df_binned = test_df.copy()
    test_df_binned["age_band"] = pd.cut(test_df_binned["age"], bins=[0, 40, 55, 70, 120])
    print("\nBy age band:")
    display(subgroup_report(test_df_binned, probs, labels, "age_band"))


### Grad-CAM: Visual Explainability

Sanity-check that the model is actually attending to retinal vasculature / optic disc region and
not to scanner artifacts, borders, or text burned into the image corner .

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# For ViT backbones, target the last transformer block's normalization layer.
target_layers = [model.backbone.blocks[-1].norm1] if hasattr(model.backbone, "blocks") else [model.backbone.layers[-1]]

def vit_reshape_transform(tensor, height=CFG.image_size // 16, width=CFG.image_size // 16):
    # Drop CLS token, reshape patch tokens back into a spatial grid.
    result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
    return result.transpose(2, 3).transpose(1, 2)

cam = GradCAM(model=model, target_layers=target_layers, reshape_transform=vit_reshape_transform)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
sample_batch = next(iter(test_loader))
for i in range(min(4, len(sample_batch["image"]))):
    input_tensor = sample_batch["image"][i:i+1].to(DEVICE)
    label = int(sample_batch["label"][i].item())
    grayscale_cam = cam(input_tensor=input_tensor, targets=[BinaryClassifierOutputTarget(0)])[0]

    # De-normalize for visualization.
    img = sample_batch["image"][i].permute(1, 2, 0).numpy()
    img = img * np.array(RETFOUND_STD) + np.array(RETFOUND_MEAN)
    img = np.clip(img, 0, 1)

    vis = show_cam_on_image(img, grayscale_cam, use_rgb=True)
    axes[i].imshow(vis); axes[i].set_title(f"true label={label}"); axes[i].axis("off")

plt.tight_layout()
plt.savefig(f"{CFG.reports_dir}/gradcam_samples.png", dpi=150)
plt.show()

## 13. Probability Calibration: Final 0–100% Likelihood Score

Fit an **isotonic regression** calibrator on the validation set.

In [ ]:
val_metrics = evaluate(model, val_loader, criterion)
val_probs, val_labels = val_metrics["probs"], val_metrics["labels"]

calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(val_probs, val_labels)

calibrated_test_probs = calibrator.predict(probs)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
reliability_diagram(probs, labels, ax=axes[0]); axes[0].set_title("Before calibration")
ece_after = reliability_diagram(calibrated_test_probs, labels, ax=axes[1]); axes[1].set_title("After calibration")
plt.tight_layout()
plt.savefig(f"{CFG.reports_dir}/calibration_after.png", dpi=150)
plt.show()

brier_after = brier_score_loss(labels, calibrated_test_probs)
print(f"ECE before: {ece_before:.4f} → after: {ece_after:.4f}")
print(f"Brier before: {brier:.4f} → after: {brier_after:.4f}")

# Persist the calibrator for use in the backend / on-device pipeline.
import joblib
joblib.dump(calibrator, f"{CFG.exports_dir}/isotonic_calibrator.joblib")

def hypertension_likelihood_percent(raw_prob: float) -> float:
    '''Final function the backend/app calls: raw sigmoid output -> calibrated 0-100 percent.'''
    return float(np.clip(calibrator.predict([raw_prob])[0] * 100, 0, 100))

print("Example: raw_prob=0.62 ->", hypertension_likelihood_percent(0.62), "% calibrated likelihood")

## 14. Model Export: Multiple On-Device Formats

The app needs to run **entirely on-device**, across Android and iOS at minimum, so I export to every format that covers that:

| Format | Target | Notes |
|---|---|---|
| TorchScript (`.pt`) | Python/LibTorch backend fallback, server-side batch scoring | easiest to validate parity against |
| ONNX (`.onnx`) | Cross-platform intermediate, ONNX Runtime Mobile | good bridge format, also usable directly on Android via ORT |
| TFLite (`.tflite`) | Android on-device | converted via ONNX>TF>TFLite; quantize for size/latency |
| Core ML (`.mlpackage`) | iOS on-device | via `coremltools`, supports on-device Neural Engine acceleration |

I also produce a **dynamically quantized** ONNX/TFLite variant, since ViT backbones are large and mobile inference latency/battery/storage all matter here.

In [ ]:
model.eval()
dummy_input = torch.randn(1, 3, CFG.image_size, CFG.image_size, device=DEVICE)

# Wrap the model to also apply the sigmoid + calibration at inference time, so exported
# artifacts return a ready-to-display probability rather than a raw logit.
class ExportWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        logit = self.base_model(x)
        return torch.sigmoid(logit)  # calibration curve is applied as a lightweight post-process step,
                                      # kept outside the graph so it stays easy to update independently.

export_model = ExportWrapper(model).to(DEVICE).eval()

### TorchScript

In [ ]:
traced = torch.jit.trace(export_model, dummy_input)
traced.save(f"{CFG.exports_dir}/retfound_htn_model.torchscript.pt")

# Parity check
with torch.no_grad():
    orig_out = export_model(dummy_input)
    ts_out = traced(dummy_input)
    assert torch.allclose(orig_out, ts_out, atol=1e-5), "TorchScript output mismatch!"
print("TorchScript export OK, outputs match within tolerance.")

### ONNX (+ dynamic quantization)

In [ ]:
onnx_path = f"{CFG.exports_dir}/retfound_htn_model.onnx"
torch.onnx.export(
    export_model, dummy_input, onnx_path,
    input_names=["fundus_image"], output_names=["hypertension_probability"],
    dynamic_axes={"fundus_image": {0: "batch"}, "hypertension_probability": {0: "batch"}},
    opset_version=17,
)

import onnx
import onnxruntime as ort

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

ort_session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
ort_out = ort_session.run(None, {"fundus_image": dummy_input.cpu().numpy()})[0]

with torch.no_grad():
    torch_out = export_model(dummy_input).cpu().numpy()
assert np.allclose(torch_out, ort_out, atol=1e-4), "ONNX output mismatch!"
print("ONNX export OK, outputs match within tolerance.")

# Dynamic quantization (weights to int8) for a smaller / faster mobile artifact.
from onnxruntime.quantization import quantize_dynamic, QuantType
quantized_path = f"{CFG.exports_dir}/retfound_htn_model.quant.onnx"
quantize_dynamic(onnx_path, quantized_path, weight_type=QuantType.QInt8)

orig_size_mb = os.path.getsize(onnx_path) / 1e6
quant_size_mb = os.path.getsize(quantized_path) / 1e6
print(f"ONNX size: {orig_size_mb:.1f} MB -> quantized: {quant_size_mb:.1f} MB")

### TFLite (Android)

Converted through TensorFlow via `onnx-tf`. Post-training quantization (float16 or full int8 with a representative dataset) is applied for on-device size/latency.

In [ ]:
import onnx
from onnx_tf.backend import prepare
import tensorflow as tf

tf_export_dir = f"{CFG.exports_dir}/tf_saved_model"
onnx_model = onnx.load(onnx_path)
tf_rep = prepare(onnx_model)
tf_rep.export_graph(tf_export_dir)

converter = tf.lite.TFLiteConverter.from_saved_model(tf_export_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]   # dynamic-range quantization by default

# For full int8 quantization, uncomment and supply a representative sample of preprocessed images:
# def representative_dataset():
#     for i in range(100):
#         img = ...  # a preprocessed (1, 224, 224, 3) float32 sample from train_ds
#         yield [img]
# converter.representative_dataset = representative_dataset
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

tflite_model = converter.convert()
tflite_path = f"{CFG.exports_dir}/retfound_htn_model.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(f"TFLite export OK -> {tflite_path} ({os.path.getsize(tflite_path)/1e6:.1f} MB)")

### Core ML (iOS)

In [ ]:
import coremltools as ct

mlmodel = ct.convert(
    traced,
    inputs=[ct.ImageType(name="fundus_image", shape=dummy_input.shape,
                          scale=1/255.0, bias=[-m/s for m, s in zip(RETFOUND_MEAN, RETFOUND_STD)])],
    outputs=[ct.TensorType(name="hypertension_probability")],
    convert_to="mlprogram",
    minimum_deployment_target=ct.target.iOS16,
)
mlmodel.short_description = "RETFound-Green fine-tuned hypertension risk screener (on-device)."
coreml_path = f"{CFG.exports_dir}/retfound_htn_model.mlpackage"
mlmodel.save(coreml_path)
print(f"Core ML export OK -> {coreml_path}")

## 15. Model Card & Packaging for the Federated Server

In [ ]:
model_card = {
    "model_name": "RETFound-Green-HTN-Screener",
    "version": "v0.1.0-round0",
    "base_model": CFG.retfound_hf_repo,
    "task": "Binary hypertension risk screening from a single fundus photograph",
    "output": "Calibrated probability (display as 0-100% likelihood) via isotonic_calibrator.joblib",
    "input_spec": {
        "image_size": CFG.image_size,
        "channels": "RGB",
        "normalization_mean": RETFOUND_MEAN,
        "normalization_std": RETFOUND_STD,
        "preprocessing": "crop_to_fov -> CLAHE(clip=2.5,tile=8x8) -> resize -> RGB -> normalize",
    },
    "training_data": {
        "n_images": len(meta_clean),
        "n_patients": int(meta_clean[CFG.patient_id_col].nunique()),
        "n_sites": int(meta_clean[CFG.site_id_col].nunique()),
        "split_strategy": "leave-one-site-out" if USE_LEAVE_ONE_SITE_OUT else "patient-grouped stratified",
    },
    "performance_on_held_out_test": {
        "auroc": float(test_metrics["auroc"]),
        "auprc": float(test_metrics["auprc"]),
        "sensitivity_at_0.5": float(sensitivity),
        "specificity_at_0.5": float(specificity),
        "ece_after_calibration": float(ece_after),
        "brier_after_calibration": float(brier_after),
    },
    "known_limitations": [
        "Single held-out site used for test — generalization to additional new sites/cameras is unverified.",
        "Not validated against a clinical gold-standard BP measurement protocol in a prospective study.",
        "Not a substitute for diagnosis; intended as a triage/screening aid for community health workers.",
        "Subgroup performance should be re-checked whenever new demographic groups are added to training data.",
    ],
    "intended_use": "Screening triage aid on CHW-operated or patient smartphones/tablets, entirely on-device.",
    "federated_role": "Initial global model (round 0) distributed to client devices for local fine-tuning.",
}

with open(f"{CFG.exports_dir}/model_card.json", "w") as f:
    json.dump(model_card, f, indent=2)

print(json.dumps(model_card, indent=2))


In [ ]:
# Bundle everything into one folder.
bundle_dir = Path(CFG.exports_dir) / "federated_round0_bundle"
bundle_dir.mkdir(exist_ok=True)

for fname in ["retfound_htn_model.torchscript.pt", "retfound_htn_model.onnx",
              "retfound_htn_model.quant.onnx", "retfound_htn_model.tflite",
              "isotonic_calibrator.joblib", "model_card.json"]:
    src = Path(CFG.exports_dir) / fname
    if src.exists():
        shutil.copy(src, bundle_dir / fname)

shutil.copytree(Path(CFG.exports_dir) / "retfound_htn_model.mlpackage", bundle_dir / "retfound_htn_model.mlpackage",
                 dirs_exist_ok=True)

shutil.make_archive(str(Path(CFG.exports_dir) / "federated_round0_bundle"), "zip", root_dir=bundle_dir)
print(f"Bundle ready: {CFG.exports_dir}/federated_round0_bundle.zip")